In [1]:
import os
from dotenv import load_dotenv
from typing import Annotated, TypedDict, List, Dict, Any
from pypdf import PdfReader

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

# Load environment variables from .env.local
load_dotenv('../.env.local')

if not os.getenv("GEMINI_API_KEY"):
    print("WARNING: GEMINI_API_KEY not found in environment variables.")

ModuleNotFoundError: No module named 'pypdf'

In [2]:
!uv add pypdf

Resolved 2 packages in 91ms                                          
Audited 1 package in 0.07ms                                          


In [ ]:
class ExpenseItem(BaseModel):
    date: str = Field(description="Date of the transaction")
    description: str = Field(description="Description of the transaction")
    amount: float = Field(description="Amount of the expense")
    category: str = Field(description="Category of the expense (e.g., Food, Transport, Utilities)")

class IncomeItem(BaseModel):
    date: str = Field(description="Date of the transaction")
    description: str = Field(description="Description of the transaction")
    amount: float = Field(description="Amount of the income")
    source: str = Field(description="Source of the income (e.g., Salary, Refund, Interest)")

In [ ]:
class StatementState(TypedDict):
    file_path: str
    raw_text: str
    analysis: Dict[str, Any]
    error: str

def pdf_parser_node(state: StatementState) -> Dict[str, Any]:
    """Parses text from a PDF file."""
    file_path = state["file_path"]
    try:
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return {"raw_text": text}
    except Exception as e:
        return {"error": str(e)}

def extractor_node(state: StatementState) -> Dict[str, Any]:
    """Extracts structured data using Gemini."""
    if state.get("error"):
        return {}
    
    raw_text = state["raw_text"]
    
    llm = ChatGoogleGenerativeAI(
        model="gemini-1.5-flash",
        google_api_key=os.getenv("GEMINI_API_KEY"),
        temperature=0
    )
    
    structured_llm = llm.with_structured_output(BankStatementAnalysis)
    
    prompt = f"""
    Analyze the following bank statement text and extract all key expenses (with negative sign followed by the expense amount) and income transactions.
    Ignore internal transfers or credit card payments if they are duplicates of other expenses.
    Categorize expenses appropriately.
    
    BANK STATEMENT TEXT:
    {raw_text[:30000]}  # Truncate to avoid token limits if necessary, though Gemini has a large context window
    """
    
    try:
        result = structured_llm.invoke(prompt)
        return {"analysis": result.model_dump()}
    except Exception as e:
        return {"error": f"LLM Extraction Error: {str(e)}"}

# Build the Graph
workflow = StateGraph(StatementState)

workflow.add_node("parse_pdf", pdf_parser_node)
workflow.add_node("extract_data", extractor_node)

workflow.add_edge(START, "parse_pdf")
workflow.add_edge("parse_pdf", "extract_data")
workflow.add_edge("extract_data", END)

app = workflow.compile()

In [ ]:
def process_bank_statement(file_path: str):
    """Helper function to run the graph."""
    if not os.path.exists(file_path):
        return "Error: File not found."
        
    initial_state = {"file_path": file_path, "raw_text": "", "analysis": {}, "error": ""}
    result = app.invoke(initial_state)
    
    if result.get("error"):
        return f"Processing Failed: {result['error']}"
    
    return result["analysis"]

# Create a dummy PDF for testing if one doesn't exist
dummy_pdf_path = "dummy_statement.pdf"
if not os.path.exists(dummy_pdf_path):
    from reportlab.pdfgen import canvas
    c = canvas.Canvas(dummy_pdf_path)
    c.drawString(100, 750, "Bank Statement - Jan 2024")
    c.drawString(100, 730, "01/01/2024 Salary Credit +$5000.00")
    c.drawString(100, 710, "05/01/2024 Walmart Grocery -$150.25")
    c.drawString(100, 690, "10/01/2024 Netflix Subscription -$15.99")
    c.drawString(100, 670, "15/01/2024 Electric Bill -$120.50")
    c.drawString(100, 650, "20/01/2024 Freelance Payment +$800.00")
    c.save()
    print(f"Created dummy PDF at {dummy_pdf_path}")

# Run the processing
output = process_bank_statement(dummy_pdf_path)
print("Extraction Result:")
import json
print(json.dumps(output, indent=2))